# Day K4 — EEG를 MNE로: 배열을 Raw로, 전극 위치, theta·beta 파워, 그리고 걷는 중 EEG의 함정

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/K-multimodal-fog/day-K4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/K-multimodal-fog/day-K4.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q mne

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 합성 EEG를 MNE Raw로 만들기

In [ ]:
import numpy as np
import mne

# ---- 공통 합성 뼈대 (K1–K5 같음) ----
fs = 500                                   # 500행 = 1초
t = np.arange(0, 60, 1/fs)                 # 60초
label = ((t >= 25) & (t < 35)).astype(int) # 25–35초가 FOG(가정)
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)

# ---- 오늘 덧붙이는 합성 EEG 3채널 (단위: µV, 모양은 모두 설명용 가정) ----
names = ["Cz", "FC1", "FC2"]
brain = np.cumsum(rng.standard_normal((3, t.size)), axis=1) * 0.05    # 느리게 출렁이는 배경
brain -= brain.mean(axis=1, keepdims=True)
alpha = 4 * np.sin(2*np.pi*10*t)                                     # 10 Hz 리듬
theta = np.where(label == 1, 6.0, 2.0) * np.sin(2*np.pi*6*t)         # 가정: FOG 중 theta가 커짐
muscle = np.where(label == 0, 5.0, 1.0) * rng.standard_normal((3, t.size))  # 가정: 걷는 동안 근육·움직임 잡음
eeg_uv = brain + alpha + theta + muscle                              # (채널 3, 표본 30000)

info = mne.create_info(ch_names=names, sfreq=fs, ch_types="eeg")
raw = mne.io.RawArray(eeg_uv * 1e-6, info, verbose=False)          # MNE는 볼트(V) 단위를 기대한다
print(raw)
print("채널:", raw.ch_names, "| fs:", raw.info["sfreq"], "Hz | 길이:", raw.n_times / fs, "초")
print("Cz 첫 3표본 (V):", raw.get_data(picks="Cz")[0, :3])
print("Cz 표준편차: %.1f µV" % (raw.get_data(picks="Cz").std() * 1e6))

### 예시 2 — 실제 파일의 채널 이름으로 montage 붙이기와 채널 확인

In [ ]:
import numpy as np
import mne

# 실제 과제 파일의 EEG 25열 이름(저자 코드 config.py 순서, K1 참고)
EEG = ["FP1","FP2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","P7","P8",
       "Fz","Cz","Pz","FC1","FC2","CP1","CP2","FC5","FC6","CP5","CP6"]
info = mne.create_info(EEG, sfreq=500, ch_types="eeg")
raw = mne.io.RawArray(np.zeros((len(EEG), 1000)), info, verbose=False)   # 모양만 보는 빈 신호

montage = mne.channels.make_standard_montage("colin27_1020")   # MNE 1.12 이하에서는 "standard_1020"
try:
    raw.set_montage(montage)                                    # 기본값: 대소문자까지 똑같아야 한다
except ValueError as e:
    print("오류: montage에서 못 찾은 채널 →", str(e).split("\n\n")[1])
raw.set_montage(montage, match_case=False)                      # FP1 = Fp1 로 맞춘다
pos = raw.get_montage().get_positions()["ch_pos"]
print("위치가 붙은 채널 수:", len(pos), "| Cz 좌표(m):", np.round(pos["Cz"], 3))

want = ["Cz", "FC1", "FC2", "TP9", "TP10", "FCz"]
have = [ch for ch in want if ch in raw.ch_names]
print("원하는 채널:", want)
print("파일에 있는 채널:", have, "| 없는 채널:", sorted(set(want) - set(have)))
sub = raw.copy().pick(have)
print(sub.ch_names)

### 예시 3 — theta·beta 절대·상대 파워, 그리고 근육 잡음이 만드는 가짜 변화

예시 1의 코드에 이어서 실행합니다(같은 노트북에서 `raw`, `fs`를 그대로 씀).

In [ ]:
import numpy as np

def band_powers(psd, freqs):
    """PSD(…, 주파수)에서 절대 파워(µV²)와 상대 파워(1–45 Hz 합 대비)."""
    df = freqs[1] - freqs[0]
    bp = lambda lo, hi: psd[..., (freqs >= lo) & (freqs < hi)].sum(-1) * df * 1e12   # V² → µV²
    total = bp(1, 45)
    return {"theta": bp(4, 8), "beta": bp(13, 30), "rel_theta": bp(4, 8) / total,
            "rel_beta": bp(13, 30) / total, "hi_60_100": bp(60, 100)}

for name, (t0, t1) in {"걷기 5–15 s": (5, 15), "FOG 25–35 s": (25, 35)}.items():
    spec = raw.compute_psd(method="welch", fmin=1, fmax=100, tmin=t0, tmax=t1 - 1/fs,
                           n_fft=1000, picks=["Cz", "FC1", "FC2"], verbose=False)
    psd, freqs = spec.get_data(return_freqs=True)      # psd: (채널 3, 주파수), 단위 V²/Hz
    b = {k: v.mean() for k, v in band_powers(psd, freqs).items()}   # 세 채널 평균
    print(f"{name}: theta {b['theta']:6.1f} µV²  beta {b['beta']:6.1f} µV²  "
          f"상대 theta {b['rel_theta']:.3f}  상대 beta {b['rel_beta']:.3f}  60–100 Hz {b['hi_60_100']:6.1f} µV²")
print("주파수 간격:", freqs[1] - freqs[0], "Hz")

# 1–45 Hz로 거른 뒤에도 beta 안의 '근육 몫'은 남는다
filt = raw.copy().filter(l_freq=1.0, h_freq=45.0, verbose=False)
for name, (t0, t1) in {"걷기": (5, 15), "FOG": (25, 35)}.items():
    spec = filt.compute_psd(method="welch", fmin=1, fmax=100, tmin=t0, tmax=t1 - 1/fs, n_fft=1000, verbose=False)
    psd, freqs = spec.get_data(return_freqs=True)
    b = {k: v.mean() for k, v in band_powers(psd, freqs).items()}
    print(f"필터 뒤 {name}: 상대 beta {b['rel_beta']:.3f}  60–100 Hz {b['hi_60_100']:.2f} µV²")

## 직접 해보기

**문제 1 — 단위 맞춰 Raw 만들기.** 예시 1에서 아래 줄의 빈칸을 채워 Cz 표준편차가 6.6 µV로 나오게 하세요.

In [ ]:
info = mne.create_info(ch_names=names, sfreq=fs, ch_types="eeg")
raw = mne.io.RawArray(____, info, verbose=False)
print("Cz 표준편차: %.1f µV" % (raw.get_data(picks="Cz").std() * 1e6))

In [ ]:
raw = mne.io.RawArray(eeg_uv * 1e-6, info, verbose=False)

**문제 2 — 원하는 채널 중 있는 것만 고르기.** 예시 2에서 빈칸을 채우세요.

In [ ]:
want = ["Cz", "FC1", "FC2", "TP9", "TP10", "FCz"]
have = [ch for ch in want if ____]
print("파일에 있는 채널:", have)

In [ ]:
have = [ch for ch in want if ch in raw.ch_names]

**문제 3 — 상대 beta.** 예시 1 다음에 실행합니다. 걷기 구간(5–15초)의 채널별 상대 beta를 구하세요.

In [ ]:
spec = raw.compute_psd(method="welch", fmin=1, fmax=100, tmin=5, tmax=15 - 1/fs, n_fft=1000, verbose=False)
psd, freqs = spec.get_data(return_freqs=True)
beta = psd[:, (freqs >= 13) & (freqs < 30)].sum(axis=1)
total = psd[:, (freqs >= 1) & (freqs < 45)].sum(axis=1)
rel_beta = ____
print("상대 beta (Cz, FC1, FC2):", np.round(rel_beta, 3))

In [ ]:
rel_beta = beta / total

**문제 4 — window마다 근육 점검.** 2초 window(1초 이동)로 잘라 상대 beta와 60–100 Hz 파워의 순위 상관(Spearman)을 구하세요. `pandas`가 필요합니다.

In [ ]:
import pandas as pd
ep = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=1.0, preload=True, verbose=False)
p, f = ep.compute_psd(method="welch", fmin=1, fmax=100, n_fft=500, n_overlap=250, verbose=False).get_data(return_freqs=True)
bp = lambda lo, hi: p[..., (f >= lo) & (f < hi)].sum(-1).mean(-1)     # (window,) 세 채널 평균
tab = pd.DataFrame({"rel_beta": bp(13, 30) / bp(1, 45), "hi_60_100": bp(60, 100)})
rho = tab["rel_beta"].corr(tab["hi_60_100"], method=____)
print("window %d개, Spearman rho = %.2f" % (len(tab), rho))

In [ ]:
rho = tab["rel_beta"].corr(tab["hi_60_100"], method="spearman")

**문제 5 (실제 데이터) — 기준 전극을 바꾸면?** 아래 "실제 데이터 1"을 실행한 뒤, 평균 기준(average reference)으로 바꿔 C3–C4 상관을 비교하세요.

In [ ]:
r = lambda R: np.corrcoef(R.get_data(picks="C3")[0], R.get_data(picks="C4")[0])[0, 1]
avg = raw.copy().set_eeg_reference(____, verbose=False)
print("C3–C4 상관: 원래 %.2f → 평균 기준 %.2f" % (r(raw), r(avg)))

In [ ]:
avg = raw.copy().set_eeg_reference("average", verbose=False)

### 실제 데이터로 해보기 (선택)

**1. 과제 파일 EEG를 Raw로, 그리고 README의 주장 확인하기**

In [ ]:
import numpy as np
import pandas as pd
import mne

PATH = "data/xuanwu/Filtered Data/003/task_1.txt"        # 머리글 없는 쉼표 구분, 61열 (K1)
EEG = ["FP1","FP2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","P7","P8",
       "Fz","Cz","Pz","FC1","FC2","CP1","CP2","FC5","FC6","CP5","CP6"]
IMU = [f"{s}_{q}" for s in ["LShank","RShank","Waist","Arm"] for q in ["ax","ay","az","gx","gy","gz","x7"]]
COLS = ["idx","clock"] + EEG + ["c27","c28","c29","c30","c31"] + IMU + ["label"]   # K1의 열 이름
d = pd.read_csv(PATH, header=None, names=COLS, usecols=EEG + ["label"])   # 필요한 열만 읽는다
eeg_uv = d[EEG].to_numpy().T                             # (25, 표본 수)
y = d["label"].to_numpy().astype(int)

# 단위는 README에 없다. 값의 크기(SD 10–40)가 µV 규모라서 µV로 가정하고 V로 바꾼다.
raw = mne.io.RawArray(eeg_uv * 1e-6, mne.create_info(EEG, 500, "eeg"), verbose=False)
raw.set_montage(mne.channels.make_standard_montage("colin27_1020"), match_case=False)
print(raw)
print("EEG 열 수:", len(raw.ch_names), "| README 표의 TP9·TP10·IO가 EEG 열에 있나:",
      [c for c in ["TP9", "TP10", "IO"] if c in raw.ch_names])
print("C3–C4 상관: %.2f (README의 'C4, C4'가 같은 열을 두 번 넣은 것이라면 1.00)"
      % np.corrcoef(raw.get_data(picks="C3")[0], raw.get_data(picks="C4")[0])[0, 1])

# 전처리 흔적 확인: Cz 전체 PSD에서 대역별 몫
spec = raw.compute_psd(method="welch", fmin=0, fmax=250, n_fft=1000, picks="Cz", verbose=False)
p, f = spec.get_data(return_freqs=True)
share = lambda lo, hi: 100 * p[0, (f >= lo) & (f < hi)].sum() / p[0].sum()
print("Cz 파워 몫(%%): <1 Hz %.1f | 1–45 Hz %.1f | 45–49 Hz %.2f | 49–51 Hz %.3f | 51–100 Hz %.2f | 100–250 Hz %.3f"
      % (share(0, 1), share(1, 45), share(45, 49), share(49, 51), share(51, 100), share(100, 251)))

**2. (원본 Raw Data를 받은 경우) 원래 기록은 몇 채널, 몇 Hz였나**

In [ ]:
import glob, warnings
import mne

vhdr = glob.glob("data/xuanwu_raw/Raw/003/*.vhdr")[0]       # 파일 이름은 기록마다 제각각이라 glob으로 찾는다
with warnings.catch_warnings(record=True) as wl:
    warnings.simplefilter("always")
    rr = mne.io.read_raw_brainvision(vhdr, preload=False, verbose=False)
for w in wl:
    print("경고:", str(w.message)[:75])
print("원본 fs:", rr.info["sfreq"], "Hz | 채널 수:", len(rr.ch_names), "| 길이: %.0f초" % (rr.n_times / rr.info["sfreq"]))
print("두피 EEG가 아닌 이름:", [c for c in rr.ch_names if c.startswith(("EMG", "IO", "TP"))])

**3. window마다 theta·beta, FOG와 비교 (환자 003)** — 1번 코드에 이어서 실행합니다.

In [ ]:
# ---- 2초 window(1초 이동)로 자르고, K2의 다수결 규칙으로 window 라벨 ----
def window_label(y_win, rule="majority"):          # K2에서 만든 함수
    if rule == "majority": return int(y_win.mean() >= 0.5)
    if rule == "any":      return int(y_win.max() == 1)
    if rule == "center":   return int(y_win[len(y_win) // 2])

epochs = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=1.0, preload=True, verbose=False)
starts = epochs.events[:, 0]                       # 각 window의 시작 표본 번호
wy = np.array([window_label(y[s:s + 1000]) for s in starts])

spec = epochs.compute_psd(method="welch", fmin=1, fmax=100, n_fft=500, n_overlap=250,
                          picks=["Cz", "FC1", "FC2"], verbose=False)
psd, freqs = spec.get_data(return_freqs=True)      # (window, 채널 3, 주파수)
df = freqs[1] - freqs[0]
bp = lambda lo, hi: psd[..., (freqs >= lo) & (freqs < hi)].sum(-1).mean(-1) * df * 1e12   # 세 채널 평균, µV²
tab = pd.DataFrame({"fog": wy, "theta": bp(4, 8), "beta": bp(13, 30),
                    "rel_theta": bp(4, 8) / bp(1, 45), "rel_beta": bp(13, 30) / bp(1, 45),
                    "hi_60_100": bp(60, 100)})
print("window 수:", len(tab), "| FOG window:", int(tab.fog.sum()))
print(tab.groupby("fog").median().round(3).to_string())
print("Spearman(상대 beta, 60–100 Hz 파워) = %.2f" % tab["rel_beta"].corr(tab["hi_60_100"], method="spearman"))

**4. 모든 환자: 환자별 중앙값과 근육 점검**

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import mne

DATA = "data/xuanwu/Filtered Data"
EEG = ["FP1","FP2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","P7","P8",
       "Fz","Cz","Pz","FC1","FC2","CP1","CP2","FC5","FC6","CP5","CP6"]
IMU = [f"{s}_{q}" for s in ["LShank","RShank","Waist","Arm"] for q in ["ax","ay","az","gx","gy","gz","x7"]]
COLS = ["idx","clock"] + EEG + ["c27","c28","c29","c30","c31"] + IMU + ["label"]
PICK = ["Cz", "FC1", "FC2"]
rows = []
for path in sorted(glob.glob(os.path.join(DATA, "**", "task_*.txt"), recursive=True)):
    parts = os.path.relpath(path, DATA).split(os.sep)  # ['003', 'task_1.txt'] 또는 ['008', 'OFF_1', 'task_1.txt']
    d = pd.read_csv(path, header=None, names=COLS, usecols=PICK + ["label"])
    raw = mne.io.RawArray(d[PICK].to_numpy().T * 1e-6, mne.create_info(PICK, 500, "eeg"), verbose=False)
    y = d["label"].to_numpy().astype(int)
    ep = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=1.0, preload=True, verbose=False)
    spec = ep.compute_psd(method="welch", fmin=1, fmax=100, n_fft=500, n_overlap=250, verbose=False)
    psd, freqs = spec.get_data(return_freqs=True)
    bp = lambda lo, hi: psd[..., (freqs >= lo) & (freqs < hi)].sum(-1).mean(-1)
    rows.append(pd.DataFrame({"patient": parts[0], "fog": [int(y[s:s + 1000].mean() >= 0.5) for s in ep.events[:, 0]],
                              "rel_theta": bp(4, 8) / bp(1, 45), "rel_beta": bp(13, 30) / bp(1, 45),
                              "hi_60_100": bp(60, 100) * 1e12}))
w = pd.concat(rows, ignore_index=True)

out = []
for pid, g in w.groupby("patient"):
    r = {"patient": pid, "windows": len(g), "fog_win": int(g.fog.sum())}
    for c in ["rel_theta", "rel_beta"]:
        m = g.groupby("fog")[c].median()
        r[c + "_nonFOG"], r[c + "_FOG"] = m.get(0, np.nan), m.get(1, np.nan)
    r["rho_beta_hi"] = g["rel_beta"].corr(g["hi_60_100"], method="spearman")
    out.append(r)
res = pd.DataFrame(out)
print(res.round(3).to_string(index=False))
both = res.dropna()
print("FOG window가 있는 환자 %d명 중 FOG에서 상대 theta가 더 큰 환자 %d명, 상대 beta가 더 작은 환자 %d명"
      % (len(both), (both.rel_theta_FOG > both.rel_theta_nonFOG).sum(), (both.rel_beta_FOG < both.rel_beta_nonFOG).sum()))